# Wikipedia Scraping for MEP Data

This notebook provides one primary, rerunnable workflow to scrape native-language and English Wikipedia pages for MEPs, plus archived legacy variants kept for reference.

## How to use this notebook

1. Run cells from top to bottom.
2. Update only the path setup variables (`USERNAME`, `PROJECT_FOLDER_NAME`, and optional file names) if your folder structure changes.
3. The main outputs are saved in the project `outputs/tables` folder.

The archived code at the end is intentionally non-executable and preserved for traceability.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [1]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import csv
import difflib
import logging
import re
from getpass import getpass as _getpass
import os
import sys
import time
import unicodedata
from urllib.parse import quote
import pandas as pd
import requests

# Ensure project root is importable in notebook kernels.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src").exists() and (candidate / "README.md").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from src.config import PROJECT_ROOT, ensure_default_directories

ensure_default_directories()


## 1) Environment setup

Installs the required Python packages for this notebook (`pandas`, `openpyxl`, `requests`).
If packages are already installed, this cell can be skipped.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [18]:
WIKIPEDIA_DIR = PROJECT_ROOT / "3 Wikipedia"
MEP_LIST_DIR = PROJECT_ROOT / "2 MEP List"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = WIKIPEDIA_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

for folder in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR, TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

INPUT_CSV = MEP_LIST_DIR / "MEP_2009_2019.csv"

OUTPUT_CSV = TABLES_DIR / "mep_wikipedia_final.csv"
OUTPUT_XLSX = TABLES_DIR / "mep_wikipedia_final.xlsx"

assert PROJECT_ROOT.exists(), f"Project root does not exist: {PROJECT_ROOT}"
assert INPUT_CSV.exists(), f"Input CSV file not found: {INPUT_CSV}"

print(f"Project root: {PROJECT_ROOT}")
print(f"Input CSV:    {INPUT_CSV}")
print(f"CSV output:   {OUTPUT_CSV}")
print(f"XLSX output:  {OUTPUT_XLSX}")

Project root: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics
Input CSV:    C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\2 MEP List\MEP_2009_2019.csv
CSV output:   C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\3 Wikipedia\outputs\tables\mep_wikipedia_final.csv
XLSX output:  C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\3 Wikipedia\outputs\tables\mep_wikipedia_final.xlsx


In [20]:
# -------- Inspect input and output files --------

# Check which files exist and where Python is looking for them.
files_to_check = {
    "Input CSV": INPUT_CSV,
    "Output CSV": OUTPUT_CSV,
    "Output XLSX": OUTPUT_XLSX,
}

print("\nFile existence check:")
for label, path in files_to_check.items():
    print(f"{label}:")
    print(f"  Path:   {path}")
    print(f"  Exists: {path.exists()}")
    if path.exists():
        print(f"  Size:   {path.stat().st_size:,} bytes")


# Inspect the input CSV structure.
if INPUT_CSV.exists():
    print("\nInput CSV preview:")

    input_df = pd.read_csv(INPUT_CSV, dtype=str)

    print(f"Rows:    {len(input_df)}")
    print(f"Columns: {list(input_df.columns)}")

    display(input_df.head(10))


# Inspect the existing output file, if it exists.
# Important: this follows the same priority as your scraper:
# it checks XLSX first, then CSV.
if OUTPUT_XLSX.exists():
    print("\nExisting output loaded from XLSX:")
    existing_df = pd.read_excel(OUTPUT_XLSX, dtype=str)

elif OUTPUT_CSV.exists():
    print("\nExisting output loaded from CSV:")
    existing_df = pd.read_csv(OUTPUT_CSV, dtype=str)

else:
    print("\nNo existing output file found.")
    existing_df = pd.DataFrame()


# Show output structure and preview if an output file was found.
if not existing_df.empty:
    print(f"Rows:    {len(existing_df)}")
    print(f"Columns: {list(existing_df.columns)}")

    display(existing_df.head(10))

    # Useful for debugging skip logic.
    if "name" in existing_df.columns:
        print("\nFirst 20 existing names:")
        display(existing_df["name"].dropna().astype(str).head(20).to_frame())


File existence check:
Input CSV:
  Path:   C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\2 MEP List\MEP_2009_2019.csv
  Exists: True
  Size:   49,778 bytes
Output CSV:
  Path:   C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\3 Wikipedia\outputs\tables\mep_wikipedia_final.csv
  Exists: False
Output XLSX:
  Path:   C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\3 Wikipedia\outputs\tables\mep_wikipedia_final.xlsx
  Exists: True
  Size:   34,114,287 bytes

Input CSV preview:
Rows:    1918
Columns: ['name', 'country']


,name,country
0,BARBAGALLO Giovanni,Italy
1,CRUZ Joaquim,Portugal
2,DIETRICH Roberto,Romania
3,GKIKOPOULOS Georgios,Greece
4,HIDVÉGHI Balázs,Hungary
5,KARI Rina Ronja,Denmark
6,KOEWIUS Annette,Germany
7,REVAULT D'ALLONNES BONNEFOY Christine,France
8,ABELA BALDACCHINO Claudette,Malta
9,BALDINI Marino,Croatia



Existing output loaded from XLSX:
Rows:    1918
Columns: ['name', 'wikipedia_language', 'page_title', 'page_url', 'wikipedia_content_html', 'wikipedia_content_wikitext', 'page_title_en', 'page_url_en', 'wikipedia_content_html_en', 'wikipedia_content_wikitext_en', 'Unnamed: 10', 'Country']


,name,wikipedia_language,page_title,page_url,wikipedia_content_html,wikipedia_content_wikitext,page_title_en,page_url_en,wikipedia_content_html_en,wikipedia_content_wikitext_en,Unnamed: 10,Country
0,BARBAGALLO Giovanni,it,Giovanni Barbagallo,https://it.wikipedia.org/wiki/Giovanni_Barbagallo,"<div class=""mw-content-ltr mw-parser-output"" l...",{{disambiguazione}}\n\n* '''[[Giovanni Barbaga...,NaN,NaN,NaN,NaN,NaN,IT
1,CRUZ Joaquim,pt,Joaquim Cruz,https://pt.wikipedia.org/wiki/Joaquim_Cruz,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Info/esporte/atleta\n|olimpico = sim\n|nome ...,Joaquim Cruz,https://en.wikipedia.org/wiki/Joaquim_Cruz,"<div class=""mw-content-ltr mw-parser-output"" l...",{{short description|Brazilian middle-distance ...,NaN,PT
2,DIETRICH Roberto,ro,Dietrich Brüggemann,https://ro.wikipedia.org/wiki/Dietrich_Br%C3%B...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Infocaseta Actor}}\n'''Dietrich Brüggemann''...,Dietrich Brüggemann,https://en.wikipedia.org/wiki/Dietrich_Br%C3%B...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{short description|German film director}}\n{{...,NaN,RO
3,GKIKOPOULOS Georgios,el,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GR
4,HIDVÉGHI Balázs,hu,Hidvéghi Balázs,https://hu.wikipedia.org/wiki/Hidv%C3%A9ghi_Ba...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Személy infobox\n| kép = Hidvéghi Balázs.jpg...,Balázs Hidvéghi,https://en.wikipedia.org/wiki/Bal%C3%A1zs_Hidv...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{short description|Hungarian politician}}\n{{...,NaN,HU
5,KARI Rina Ronja,da,Rina Ronja Kari,https://da.wikipedia.org/wiki/Rina_Ronja_Kari,"<div class=""mw-content-ltr mw-parser-output"" l...",{{forældet|Ikke genvalgt}}\n{{Infoboks leder\n...,Rina Ronja Kari,https://en.wikipedia.org/wiki/Rina_Ronja_Kari,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Short description|Danish politician}}\n{{Inf...,NaN,DK
6,KOEWIUS Annette,de,Annette Koewius,https://de.wikipedia.org/wiki/Annette_Koewius,"<div class=""mw-content-ltr mw-parser-output"" l...",'''Annette Koewius''' (* [[22. September]] [[1...,NaN,NaN,NaN,NaN,NaN,DE
7,REVAULT D'ALLONNES BONNEFOY Christine,fr,Revault d'Allonnes,https://fr.wikipedia.org/wiki/Revault_d%27Allo...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{homonymie}}\n\n== Nom de famille ==\n{{Nom d...,NaN,NaN,NaN,NaN,NaN,FR
8,ABELA BALDACCHINO Claudette,mt,NaN,NaN,NaN,NaN,Claudette Abela Baldacchino,https://en.wikipedia.org/wiki/Claudette_Abela_...,"<div class=""mw-content-ltr mw-parser-output"" l...",{{short description|Maltese politician}}\n\n\n...,NaN,MT
9,BALDINI Marino,hr,Marino Baldini,https://hr.wikipedia.org/wiki/Marino_Baldini,"<div class=""mw-content-ltr mw-parser-output"" l...","'''Marino Baldini''' ([[Poreč]], [[12. srpnja]...",Marino Baldini,https://en.wikipedia.org/wiki/Marino_Baldini,"<div class=""mw-content-ltr mw-parser-output"" l...",{{Short description|Croatian politician (born ...,NaN,HR



First 20 existing names:


,name
0,BARBAGALLO Giovanni
1,CRUZ Joaquim
2,DIETRICH Roberto
3,GKIKOPOULOS Georgios
4,HIDVÉGHI Balázs
5,KARI Rina Ronja
6,KOEWIUS Annette
7,REVAULT D'ALLONNES BONNEFOY Christine
8,ABELA BALDACCHINO Claudette
9,BALDINI Marino


In [3]:
# Prompt for key only if it is missing in the current notebook kernel.

existing_key = (os.environ.get("OPENROUTER_API_KEY") or "").strip()
if existing_key:
    API_KEY = existing_key
    print("OPENROUTER_API_KEY already set in this kernel session.")
else:
    entered_key = _getpass("Enter OPENROUTER_API_KEY (input hidden): ").strip()
    if entered_key:
        os.environ["OPENROUTER_API_KEY"] = entered_key
        API_KEY = entered_key
        print("OPENROUTER_API_KEY stored in this kernel session.")
    else:
        API_KEY = ""
        print("No key entered. LLM-related cells will fail until a key is provided.")

OPENROUTER_API_KEY stored in this kernel session.


In [4]:
# ------------------------------
# OpenRouter API key pre-check
# ------------------------------
import os

key = (
    (globals().get("API_KEY", "") or "").strip().strip('"').strip("'")
    or (os.environ.get("OPENROUTER_API_KEY") or "").strip()
    or (os.environ.get("OPENROUTER_KEY") or "").strip()
)

print("Key present in kernel:", bool(key))
if key:
    print("Key format looks OpenRouter-like:", key.startswith("sk-or-v1-"))
    try:
        resp = requests.get(
            "https://openrouter.ai/api/v1/models",
            headers={"Authorization": f"Bearer {key}"},
            timeout=30,
        )
        print("Auth test status:", resp.status_code)
        if resp.status_code == 401:
            print("Unauthorized: key is invalid/revoked, or not from OpenRouter.")
        elif resp.ok:
            print("Auth OK. You can run the LLM-related cells.")
        else:
            print("Auth failed with response:", resp.text[:300])
    except Exception as e:
        print("Auth request failed:", e)
else:
    print("No key found. Run the API key prompt cell first, then rerun this test cell.")

Key present in kernel: True
Key format looks OpenRouter-like: True
Auth test status: 200
Auth OK. You can run the LLM-related cells.


## 2) Main scraper logic

The next cell contains the primary workflow used for final outputs.
It keeps the original analytical logic (fuzzy title resolution, native-language + English retrieval, incremental behavior), but now uses the shared path setup and cleaner configuration.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [26]:

# -------- Configuration --------

# Identifies this scraper when making requests to Wikipedia.
# Replace the email with your real contact if you share/run this seriously.
USER_AGENT = "MEP-Wiki-Scraper/1.7 (contact: you@example.com)"

# Pause between Wikipedia requests to avoid sending too many requests too quickly.
REQUEST_PAUSE_SECONDS = 0.5

# HTTP headers sent with each Wikipedia API request.
HEADERS = {"User-Agent": USER_AGENT}

# Maximum number of Wikipedia search results to consider per query.
SEARCH_LIMIT = 10

# Uses INPUT_CSV, OUTPUT_CSV, OUTPUT_XLSX from the path setup cell.
INPUT_CSV = Path(INPUT_CSV)
OUTPUT_CSV = Path(OUTPUT_CSV)
OUTPUT_XLSX = Path(OUTPUT_XLSX)

# -------- Country -> Wikipedia language --------
COUNTRY_TO_LANG = {
    "Finland": "fi",
    "Spain": "es",
    "Poland": "pl",
    "Greece": "el",
    "Malta": "mt",
    "France": "fr",
    "Germany": "de",
    "Italy": "it",
    "Portugal": "pt",
    "Netherlands": "nl",
    "Belgium": "fr",
    "Sweden": "sv",
    "Denmark": "da",
    "Czechia": "cs",
    "Czech Republic": "cs",
    "Slovakia": "sk",
    "Slovenia": "sl",
    "Austria": "de",
    "Ireland": "ga",
    "Romania": "ro",
    "Bulgaria": "bg",
    "Hungary": "hu",
    "Croatia": "hr",
    "Lithuania": "lt",
    "Latvia": "lv",
    "Estonia": "et",
    "Cyprus": "el",
    "Luxembourg": "lb",
}

# -------- Load MEPs from CSV --------

def load_meps_from_csv(path: Path, n: int = 999999) -> List[dict]:
    """
    Loads MEP names and countries from the input CSV file.

    This version expects lowercase input columns:
    - name
    - country

    Returns:
    [
        {"fullName": "ADAKTUSSON Lars", "country": "Sweden"},
        ...
    ]
    """

    # Stop if the CSV file does not exist.
    if not path.exists():
        raise FileNotFoundError(f"CSV file not found at: {path}")

    # Read the CSV file as text.
    # dtype=str prevents pandas from guessing wrong data types.
    df = pd.read_csv(path, dtype=str)

    # Optional: clean column names in case there are accidental spaces.
    df.columns = df.columns.str.strip()

    # Your input CSV must have these two lowercase columns.
    required_columns = {"name", "country"}

    if not required_columns.issubset(df.columns):
        raise KeyError(
            "Missing expected columns: 'name' and 'country'. "
            f"Present columns: {list(df.columns)}"
        )

    # Use the lowercase column names directly.
    name_col = "name"
    country_col = "country"

    # Keep only the required columns and optionally limit the number of rows.
    df = df[[name_col, country_col]].head(n).copy()

    # Clean missing values and extra spaces.
    df[name_col] = df[name_col].fillna("").astype(str).str.strip()
    df[country_col] = df[country_col].fillna("").astype(str).str.strip()

    # Drop rows where either name or country is empty.
    df = df[(df[name_col] != "") & (df[country_col] != "")]

    # Convert the dataframe into the structure used by the scraper.
    return [
        {"fullName": row[0], "country": row[1]}
        for row in df[[name_col, country_col]].itertuples(index=False, name=None)
    ]

# -------- Helpers --------
def strip_diacritics(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")


def simplify(s: str) -> str:
    s = strip_diacritics(s.lower())
    s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalise_name_case(name: str) -> str:
    parts = name.split()
    return " ".join(p.capitalize() if p.isupper() else p for p in parts)


def generate_name_variants(name: str) -> List[str]:
    variants: List[str] = []
    base = name.strip()

    def add(v: str) -> None:
        if v and v not in variants:
            variants.append(v)

    add(base)
    add(normalise_name_case(base))
    no_diac = strip_diacritics(base)
    add(no_diac)
    add(normalise_name_case(no_diac))
    add(" ".join(base.split()))

    tokens = base.split()
    if len(tokens) >= 2:
        first, last = tokens[0], tokens[-1]
        add(f"{first} {last}")
        add(f"{last} {first}")
        add(strip_diacritics(f"{first} {last}"))
        add(strip_diacritics(f"{last} {first}"))
        add(normalise_name_case(f"{first} {last}"))
        add(normalise_name_case(f"{last} {first}"))

    replacements = [
        ("Georgios", "Giorgos"),
        ("Giorgos", "Georgios"),
        ("Georgios", "Yorgos"),
        ("Yorgos", "Georgios"),
    ]
    for old, new in replacements:
        if old in base:
            add(base.replace(old, new))
            add(normalise_name_case(base.replace(old, new)))

    return variants


def mw_query(lang: str, params: Dict) -> Dict:
    base = f"https://{lang}.wikipedia.org/w/api.php"
    response = requests.get(base, params=params, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()


def mw_exact_title_if_exists(lang: str, title: str) -> Optional[str]:
    data = mw_query(
        lang,
        {
            "action": "query",
            "format": "json",
            "prop": "info",
            "redirects": "1",
            "titles": title,
            "inprop": "url",
        },
    )
    pages = data.get("query", {}).get("pages", {})
    for pid, page in pages.items():
        if str(pid).lstrip("-").isdigit() and int(pid) > 0 and "missing" not in page:
            return page.get("title")
    return None


def mw_search_candidates(lang: str, query: str, limit: int = SEARCH_LIMIT) -> List[str]:
    titles: List[str] = []

    data = mw_query(
        lang,
        {
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": query,
            "srlimit": str(limit),
            "utf8": "1",
            "srprop": "",
        },
    )
    for hit in data.get("query", {}).get("search", []):
        title = hit.get("title")
        if title and title not in titles:
            titles.append(title)

    data2 = mw_query(
        lang,
        {
            "action": "query",
            "format": "json",
            "list": "prefixsearch",
            "pssearch": query,
            "pslimit": str(limit),
        },
    )
    for hit in data2.get("query", {}).get("prefixsearch", []):
        title = hit.get("title")
        if title and title not in titles:
            titles.append(title)

    return titles


def score_title(candidate: str, target_name: str) -> float:
    a = simplify(candidate)
    b = simplify(target_name)
    score = difflib.SequenceMatcher(None, a, b).ratio()
    if a.startswith(b) or b.startswith(a):
        score += 0.15
    ta, tb = set(a.split()), set(b.split())
    if ta and tb:
        overlap = len(ta & tb) / max(len(ta), 1)
        score += 0.2 * overlap
    return min(score, 1.0)


def resolve_title_fuzzy(lang: str, name: str) -> Optional[str]:
    for candidate in generate_name_variants(name):
        exact = mw_exact_title_if_exists(lang, candidate)
        if exact:
            return exact

    pool: List[str] = []
    queries: List[str] = []
    for candidate in generate_name_variants(name):
        queries.extend([f'"{candidate}"', f'intitle:"{candidate}"', candidate])

    for query in queries:
        for title in mw_search_candidates(lang, query, limit=SEARCH_LIMIT):
            if title not in pool:
                pool.append(title)
        if len(pool) >= SEARCH_LIMIT:
            break

    if not pool:
        return None

    best = max(pool, key=lambda t: score_title(t, name))
    return best if score_title(best, name) >= 0.62 else None


def fetch_html_and_wikitext(lang: str, title: str) -> Tuple[str, str, str, Dict[str, dict]]:
    data = mw_query(
        lang,
        {
            "action": "parse",
            "page": title,
            "prop": "text|displaytitle|langlinks",
            "format": "json",
            "formatversion": "2",
            "redirects": "1",
            "lllimit": "500",
        },
    )
    if "parse" not in data:
        return "", "", "", {}

    html = data["parse"].get("text") or ""
    display_title = data["parse"].get("title") or title
    url = f"https://{lang}.wikipedia.org/wiki/{quote(display_title.replace(' ', '_'))}"

    langlinks_raw = data["parse"].get("langlinks") or []
    langlinks: Dict[str, dict] = {}
    for ll in langlinks_raw:
        code = ll.get("lang")
        if not code:
            continue
        ll_title = ll.get("title") or ll.get("*") or ""
        ll_url = ll.get("url") or (
            f"https://{code}.wikipedia.org/wiki/{quote(ll_title.replace(' ', '_'))}" if ll_title else ""
        )
        langlinks[code] = {"title": ll_title, "url": ll_url}

    data_w = mw_query(
        lang,
        {
            "action": "parse",
            "page": title,
            "prop": "wikitext",
            "format": "json",
            "formatversion": "2",
            "redirects": "1",
        },
    )
    wikitext_val = ""
    if "parse" in data_w:
        wt = data_w["parse"].get("wikitext")
        if isinstance(wt, str):
            wikitext_val = wt
        elif isinstance(wt, dict):
            wikitext_val = wt.get("*", "") or wt.get("content", "")

    return html, wikitext_val, url, langlinks


def resolve_via_english_bridge(native_lang: str, name: str) -> Optional[str]:
    en_title = resolve_title_fuzzy("en", name)
    if not en_title:
        return None
    _, _, _, langlinks = fetch_html_and_wikitext("en", en_title)
    native = langlinks.get(native_lang)
    return native["title"] if native and native.get("title") else None


def fetch_native_side(lang: Optional[str], name: str) -> Tuple[str, str, str, str, Dict[str, dict]]:
    if not lang:
        return "", "", "", "", {}
    title_native = resolve_title_fuzzy(lang, name)
    if not title_native:
        bridged = resolve_via_english_bridge(lang, name)
        if bridged:
            title_native = bridged
    if not title_native:
        return "", "", "", "", {}
    html_native, wiki_native, url_native, langlinks = fetch_html_and_wikitext(lang, title_native)
    return title_native or "", url_native or "", html_native or "", wiki_native or "", langlinks


def fetch_english_side(
    name: str,
    langlinks_from_native: Optional[Dict[str, dict]] = None,
    ) -> Tuple[str, str, str, str]:
    page_title_en = ""
    page_url_en = ""

    if langlinks_from_native and "en" in langlinks_from_native and langlinks_from_native["en"].get("title"):
        page_title_en = langlinks_from_native["en"]["title"] or ""
        page_url_en = langlinks_from_native["en"].get("url") or ""

    if not page_title_en:
        maybe_en = resolve_title_fuzzy("en", name)
        if maybe_en:
            page_title_en = maybe_en

    html_en, wiki_en = "", ""
    if page_title_en:
        html_en, wiki_en, url_en, _ = fetch_html_and_wikitext("en", page_title_en)
        if url_en:
            page_url_en = url_en

    return page_title_en or "", page_url_en or "", html_en or "", wiki_en or ""


def main() -> None:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

    # Load the input CSV with the MEPs you want to process.
    meps_from_file = load_meps_from_csv(INPUT_CSV, n=999999)

    if not meps_from_file:
        raise RuntimeError("No MEP rows loaded from CSV. Check name/country columns in the input file.")

    # Load the existing final output, if it already exists.
    # Priority: XLSX first, then CSV.
    # Important: existing rows are kept as they are.
    if OUTPUT_XLSX.exists():
        existing_df = pd.read_excel(OUTPUT_XLSX, dtype=str)
        logging.info("Loaded existing final output: %s (%d rows)", OUTPUT_XLSX, len(existing_df))

    elif OUTPUT_CSV.exists():
        existing_df = pd.read_csv(OUTPUT_CSV, dtype=str)
        logging.info("Loaded existing final output: %s (%d rows)", OUTPUT_CSV, len(existing_df))

    else:
        existing_df = pd.DataFrame(
            columns=[
                "name",
                "wikipedia_language",
                "page_title",
                "page_url",
                "wikipedia_content_html",
                "wikipedia_content_wikitext",
                "page_title_en",
                "page_url_en",
                "wikipedia_content_html_en",
                "wikipedia_content_wikitext_en",
            ]
        )
        logging.info("No existing final output found, starting fresh.")

    # Make sure the final output has a name column.
    if "name" not in existing_df.columns:
        existing_df["name"] = ""

    # Create a set of names already present in the final file.
    # These names will not be touched, replaced, or reprocessed.
    existing_names = set(
        existing_df["name"]
        .dropna()
        .astype(str)
        .str.strip()
    )

    # Keep all existing rows exactly as loaded.
    rows = existing_df.to_dict(orient="records")

    new_rows_added = 0
    skipped_rows = 0

    for mep in meps_from_file:
        name = mep["fullName"].strip()
        country = mep["country"].strip()
        lang = COUNTRY_TO_LANG.get(country)

        # If this person is already in the final file, do nothing.
        # No overwrite. No replacement. No update.
        if name in existing_names:
            logging.info("Keeping existing row for %s", name)
            skipped_rows += 1
            continue

        logging.info("Processing new person only: %s [%s -> %s + en]", name, country, lang or "N/A")

        page_title_native, page_url_native, html_native, wiki_native, langlinks_native = fetch_native_side(lang, name)

        page_title_en, page_url_en, html_en, wiki_en = fetch_english_side(
            name,
            langlinks_from_native=langlinks_native if langlinks_native else None,
        )

        # Append only genuinely new rows.
        rows.append(
            {
                "name": name,
                "wikipedia_language": lang or "",
                "page_title": page_title_native,
                "page_url": page_url_native,
                "wikipedia_content_html": html_native,
                "wikipedia_content_wikitext": wiki_native,
                "page_title_en": page_title_en,
                "page_url_en": page_url_en,
                "wikipedia_content_html_en": html_en,
                "wikipedia_content_wikitext_en": wiki_en,
            }
        )

        existing_names.add(name)
        new_rows_added += 1

        # Pause to avoid hammering Wikipedia.
        time.sleep(REQUEST_PAUSE_SECONDS)

    # Rebuild the dataframe.
    # Existing rows remain, new rows are appended at the bottom.
    out_df = pd.DataFrame(
        rows,
        columns=[
            "name",
            "wikipedia_language",
            "page_title",
            "page_url",
            "wikipedia_content_html",
            "wikipedia_content_wikitext",
            "page_title_en",
            "page_url_en",
            "wikipedia_content_html_en",
            "wikipedia_content_wikitext_en",
        ],
    )

    # Save the final file with existing rows preserved and new rows appended.
    # This rewrites the file technically, but it does not overwrite existing people with new data.
    out_df.to_csv(OUTPUT_CSV, index=False, quoting=csv.QUOTE_ALL, encoding="utf-8-sig")
    out_df.to_excel(OUTPUT_XLSX, index=False)

    print(f"Existing rows kept: {skipped_rows}")
    print(f"New rows added:     {new_rows_added}")
    print(f"Total rows saved:   {len(out_df)}")
    print(f"CSV:  {OUTPUT_CSV}")
    print(f"XLSX: {OUTPUT_XLSX}")

main()


INFO: Loaded existing final output: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\3 Wikipedia\outputs\tables\mep_wikipedia_final.xlsx (1918 rows)
INFO: Keeping existing row for BARBAGALLO Giovanni
INFO: Keeping existing row for CRUZ Joaquim
INFO: Keeping existing row for DIETRICH Roberto
INFO: Keeping existing row for GKIKOPOULOS Georgios
INFO: Keeping existing row for HIDVÉGHI Balázs
INFO: Keeping existing row for KARI Rina Ronja
INFO: Keeping existing row for KOEWIUS Annette
INFO: Keeping existing row for REVAULT D'ALLONNES BONNEFOY Christine
INFO: Keeping existing row for ABELA BALDACCHINO Claudette
INFO: Keeping existing row for BALDINI Marino
INFO: Keeping existing row for BERTOT Fabrizio
INFO: Keeping existing row for BONANINI Franco
INFO: Keeping existing row for BORISSOV Preslav
INFO: Keeping existing row for BORZAN Biljana
INFO: Keeping existing row for

Existing rows kept: 1918
New rows added:     0
Total rows saved:   1918
CSV:  C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\3 Wikipedia\outputs\tables\mep_wikipedia_final.csv
XLSX: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\3 Wikipedia\outputs\tables\mep_wikipedia_final.xlsx


## 3) Quick output checks

Run a lightweight validation on the saved table to confirm row count, column presence, and missing values in key fields before hand-in.

What this section does:
- Clarifies the purpose of this section and how it connects to the next steps.
- Summarizes expected inputs and outputs to make reruns easier to follow.


In [25]:

check_df = pd.read_excel(OUTPUT_XLSX, dtype=str) if OUTPUT_XLSX.exists() else pd.DataFrame()

print(f"Rows: {len(check_df)}")
print(f"Columns: {len(check_df.columns)}")

if not check_df.empty:
    key_cols = ["name", "wikipedia_language", "page_url", "page_url_en"]
    existing_key_cols = [c for c in key_cols if c in check_df.columns]
    missing_summary = check_df[existing_key_cols].isna().sum().sort_values(ascending=False)
    print("Missing values in key columns:")
    display(missing_summary.to_frame("missing_count"))

print(f"Saved tables folder: {TABLES_DIR}")


Rows: 1634
Columns: 10
Missing values in key columns:


,missing_count
page_url,133
page_url_en,61
wikipedia_language,0
name,0


Saved tables folder: C:\Users\Gebruiker\OneDrive - Università Commerciale Luigi Bocconi\Master 2\Semester I\Population dynamics and economics\Group work\Population\Analysis Population Dynamics\3 Wikipedia\outputs\tables
